# 🔗 Chain Migration, Advanced — conversational retrieval without the chain class

## Learning Objectives
In this notebook, you will learn:
1. **`ConversationalRetrievalChain` → LCEL** - the retired class, and what it was doing
2. **Rephrase then retrieve** - turning a follow-up question into a standalone one
3. **Composing the two halves** - `rephrase_chain | retrieval_chain`
4. **Returning source documents** - keeping the retrieved docs alongside the answer

## Prerequisites
- `langchain >= 1.4.0`, `langchain-core >= 1.6.1`, `langchain-classic >= 1.0.8`,
  plus `langchain-openai` and `langchain-chroma` (the floors in this repo's `pyproject.toml`)
- `OPENAI_API_KEY` in your project `.env`
- Notebooks `3.1_LCEL_Introduction` and `3.5_Chain_Migrations`

> Source: <https://www.udemy.com/course/langchain-in-action-develop-llm-powered-applications/>

In [ ]:
# ============================================================================
# SETUP: documents, embeddings and a retriever
# ============================================================================
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

load_dotenv()

embedding_function = OpenAIEmbeddings()

docs = [
    Document(
        page_content="the dog loves to eat pizza", metadata={"source": "animal.txt"}
    ),
    Document(
        page_content="the cat loves to eat lasagna", metadata={"source": "animal.txt"}
    ),
]


db = Chroma.from_documents(docs, embedding_function)
llm = ChatOpenAI(model="gpt-4o-mini")
retriever = db.as_retriever()

# Provider alternatives. NOTE: swapping the line below is NOT sufficient --
# the rephrase, answer and source-returning steps (cells 7, 11 and 16) each
# construct their own client inline, because they want deterministic output for
# deterministic rewriting and retrieval. Swap those too, or you end up with a
# provider-mixed notebook that still needs OPENAI_API_KEY.
# llm = ChatGroq(model="openai/gpt-oss-120b")
# llm = ChatAnthropic(model="claude-sonnet-4-5")

print(f"🤖 Model loaded: {llm.model_name}")
print("✅ Setup complete!")

In [ ]:
# ============================================================================
# SANITY CHECK: what the retriever returns
# ============================================================================
retriever.invoke("What exactly?")

---
### 🕰️ Legacy: `ConversationalRetrievalChain`

> **LangChain 1.x**: `ConversationalRetrievalChain` is retired and imports from
> `langchain-classic`. It still runs (removal is 2.0.0).
>
> The cell below triggers **at least two** deprecation warnings: one for the
> class, one for calling a chain directly (`chain({...})` → `chain.invoke({...})`,
> changed in 0.1.0). Expect more — the chain builds other deprecated chains
> internally and calls their retired `.run()` method.
>
> The framework's own suggested replacement is
> `create_history_aware_retriever` + `create_retrieval_chain`. This notebook
> instead composes the behaviour by hand with LCEL, which shows what those
> helpers actually do.

In [ ]:
# ============================================================================
# LEGACY 0.X: ConversationalRetrievalChain
# ============================================================================
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_core.prompts import ChatPromptTemplate


qa_template = """
You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer
the question. If you don't know the answer, say that you
don't know. Use three sentences maximum and keep the
answer concise.

Chat History:
{chat_history}

Other context:
{context}

Question: {question}
"""

qa_prompt = ChatPromptTemplate.from_template(qa_template)

convo_qa_chain = ConversationalRetrievalChain.from_llm(
    llm,
    retriever,
    return_source_documents=True,
    combine_docs_chain_kwargs={
        "prompt": qa_prompt,
    },
)

convo_qa_chain(
    {
        "question": "What kind of food does the cat like?",
        "chat_history": "",
    }
)

---
### ✨ The LCEL rewrite

From here the notebook composes the same behaviour by hand: a rephrase step, a
retrieval step, and an answer step, piped together. Nothing below this point
imports from `langchain-classic`.

In [ ]:
# ============================================================================
# REPHRASE STEP: turn a follow-up into a standalone question
# ============================================================================
from langchain_core.prompts.prompt import PromptTemplate

rephrase_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""
REPHRASE_TEMPLATE = PromptTemplate.from_template(rephrase_template)

In [ ]:
# ============================================================================
# REPHRASE STEP: build the chain that rewrites a follow-up into a standalone question
# ============================================================================
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser

rephrase_chain = REPHRASE_TEMPLATE | ChatOpenAI(model="gpt-4o-mini", temperature=0) | StrOutputParser()

In [ ]:
# ============================================================================
# REPHRASE STEP: try it
# ============================================================================
rephrase_chain.invoke(
    {
        "question": "No, really?",
        "chat_history": [
            HumanMessage(content="What does the dog like to eat?"),
            AIMessage(content="Thuna!"),
        ],
    }
)

---
### 💬 Answer step

The second half: take the standalone question, retrieve context for it, and
answer from that context. Note this is a plain LCEL pipe — nothing here knows
anything about chat history, because the rephrase step already handled it.

In [ ]:
# ============================================================================
# ANSWER STEP: the retrieval prompt
# ============================================================================
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
ANSWER_PROMPT = ChatPromptTemplate.from_template(template)

In [ ]:
# ============================================================================
# ANSWER STEP: retrieval chain
# ============================================================================
from langchain_core.runnables import RunnablePassthrough

retrieval_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | ANSWER_PROMPT
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
)

---
### 🔗 Composing the two chains

With both halves built, composition is one pipe: the rephrase chain's output
becomes the retrieval chain's input. This is the point where LCEL earns its
keep — the two stages stay independently testable and independently swappable.

In [ ]:
# ============================================================================
# COMPOSE: rephrase then retrieve
# ============================================================================
final_chain = rephrase_chain | retrieval_chain

In [ ]:
# ============================================================================
# COMPOSE: try the full chain
# ============================================================================
final_chain.invoke(
    {
        "question": "No, really?",
        "chat_history": [
            HumanMessage(content="What does the dog like to eat?"),
            AIMessage(content="Thuna!"),
        ],
    }
)

---
### 📎 Chat with returning documents

`ConversationalRetrievalChain` had a `return_source_documents=True` flag for this.
In LCEL there is no flag — you shape the output yourself. The step below builds a
dict carrying both the retrieved documents and the question, so the answer and its
sources come back together.

That is the trade the whole notebook is about: one less magic keyword, one more
line of code you can read.

In [ ]:
# ============================================================================
# RETURNING SOURCES: keep the documents alongside the answer
# ============================================================================
retrieved_documents = {"docs": retriever, "question": RunnablePassthrough()}
final_inputs = {
    "context": lambda x: "\n".join(doc.page_content for doc in x["docs"]),
    "question": lambda x: x["question"],
}
answer = {
    "answer": final_inputs | ANSWER_PROMPT | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser(),
    "docs": lambda x: x["docs"],
}

final_chain = rephrase_chain | retrieved_documents | answer

In [ ]:
# ============================================================================
# RETURNING SOURCES: invoke
# ============================================================================
result = final_chain.invoke(
    {
        "question": "No, really?",
        "chat_history": [
            HumanMessage(content="What does the dog like to eat?"),
            AIMessage(content="Thuna!"),
        ],
    }
)
print(result)

In [ ]:
# ============================================================================
# RETURNING SOURCES: the answer
# ============================================================================
result["answer"]

In [ ]:
# ============================================================================
# RETURNING SOURCES: the documents
# ============================================================================
result["docs"]

---
## 📝 Summary

### 1. What `ConversationalRetrievalChain` was doing
- **Key point**: rephrase the follow-up into a standalone question, retrieve on that, then answer
- **Key point**: written as LCEL, each of those three steps is visible and independently tunable

### 2. Composition
- **Key point**: `final_chain = rephrase_chain | retrieval_chain` — two chains, piped
- **Key point**: returning sources is a dict-shaped step, not a `return_source_documents=True` flag

### 3. The legacy half
- **Key point**: the class still imports from `langchain-classic`, and is kept here only as contrast

### Next Steps
- `8.2_Doc_Chains_to_LCEL_LangChain_v1.ipynb` — document-combining chains as LCEL
- Phase 4 (`04_Retrieval_and_RAG/`) for retrieval strategy itself, rather than its plumbing